# 🔥 EnerGIS Scenario Studio
Interaktive Optimierung von industriellen Energiesystemen

## 🎯 Quick Start
1. Kernel auswählen (oben rechts): Python 3.11+ empfohlen
2. Alle Zellen nacheinander mit **Shift+Enter** ausführen
3. Bei Fehlern: Traceback lesen und Config/Daten prüfen

---

## 📦 Setup & Imports

In [ ]:
# System-Path Setup (damit energis gefunden wird)
import sys
from pathlib import Path

# Automatisch zum Projekt-Root navigieren
notebook_path = Path().resolve()
project_root = notebook_path.parent if notebook_path.name == 'notebooks' else notebook_path

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"📁 Projekt-Root: {project_root}")
print(f"✅ Python-Path erweitert")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from datetime import datetime

# EnerGIS-Module
try:
    from energis.run.orchestrator import run_all
    from energis.config.merge import load_and_merge
    print("✅ EnerGIS-Module erfolgreich geladen")
except ImportError as e:
    print(f"❌ FEHLER beim Import: {e}")
    print("💡 Tipp: Stelle sicher, dass du im Projekt-Root bist")
    raise

# Settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Display-Optionen
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("\n🎨 Matplotlib & Pandas konfiguriert")
print(f"📊 Pandas Version: {pd.__version__}")
print(f"🐍 Python Version: {sys.version.split()[0]}")

---
## ⚙️ Konfiguration

In [ ]:
# Config-Pfade (relativ zum Projekt-Root)
cfg_paths = [
    "configs/base.yaml",
    "configs/tech_catalog.yaml",
    "configs/sites/default.site.yaml",
    "configs/systems/baseline.system.yaml",
    "configs/scenarios/pf_then_rh.workflow.scenario.yaml",
]

# Prüfen ob Dateien existieren
print("📋 Konfigurationsdateien:")
all_exist = True
for p in cfg_paths:
    full_path = project_root / p
    exists = full_path.exists()
    symbol = "✅" if exists else "❌"
    print(f"  {symbol} {p}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("❌ Nicht alle Config-Dateien gefunden!")

# Optional: Overrides für Quick-Tests
overrides = None  # Oder z.B.: {"run": {"solver": "glpk"}}

print("\n✅ Konfiguration OK")

In [ ]:
# Config-Vorschau (optional)
cfg_preview = load_and_merge(cfg_paths)

print("🔍 Config-Vorschau:")
print(f"  Solver:        {cfg_preview.get('run', {}).get('solver', 'N/A')}")
print(f"  Zeitschritt:   {cfg_preview.get('run', {}).get('dt_h', 'N/A')} h")
print(f"  CO2-Preis:     {cfg_preview.get('costs', {}).get('co2_price_eur_per_t', 'N/A')} EUR/t")
print(f"  Input-Datei:   {cfg_preview.get('site', {}).get('input_xlsx', 'N/A')}")
print(f"  Jahr:          {cfg_preview.get('site', {}).get('year_target', 'N/A')}")

# Systemkomponenten
sys_cfg = cfg_preview.get('system', {})
n_hp = len([hp for hp in sys_cfg.get('heat_pumps', []) if hp.get('enabled', True)])
n_gen = len([k for k,v in sys_cfg.get('generators', {}).items() if v.get('enabled', False)])
storage = sys_cfg.get('storage', {}).get('enabled', False)

print(f"\n🏭 Systemkomponenten:")
print(f"  Wärmepumpen:   {n_hp}")
print(f"  Generatoren:   {n_gen}")
print(f"  Speicher:      {'Ja' if storage else 'Nein'}")

---
## 🚀 Optimierung ausführen

In [ ]:
%%time
# Hauptlauf mit Error-Handling
print("="*70)
print("▶ STARTE OPTIMIERUNG")
print("="*70)
print(f"⏰ Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    res = run_all(cfg_paths, overrides=overrides)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH ABGESCHLOSSEN")
    print("="*70)
    print(f"\n📊 Export:     {res['scenario_xlsx']}")
    print(f"📁 Verzeichnis: {res['outdir']}")
    print(f"\n💰 Kosten:")
    for key, val in res['costs'].items():
        if isinstance(val, (int, float)):
            print(f"    {key:20s}: {val:,.2f}")
        else:
            print(f"    {key:20s}: {val}")
    
    optimization_successful = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER BEI DER OPTIMIERUNG")
    print("="*70)
    print(f"\n🔴 Fehler: {str(e)}\n")
    
    import traceback
    print("📋 Vollständiger Traceback:")
    traceback.print_exc()
    
    res = None
    optimization_successful = False
    
    print("\n💡 Troubleshooting:")
    print("  1. Prüfe ob Import_Data.xlsx existiert")
    print("  2. Prüfe Solver-Installation (gurobi/glpk)")
    print("  3. Prüfe ob alle Dependencies installiert sind")

---
## 📊 Ergebnisse laden

In [ ]:
if optimization_successful and res:
    # Excel laden
    try:
        ts = pd.read_excel(res['scenario_xlsx'], sheet_name='timeseries', index_col=0)
        costs_df = pd.read_excel(res['scenario_xlsx'], sheet_name='costs', index_col=0)
        meta_df = pd.read_excel(res['scenario_xlsx'], sheet_name='meta')
        
        print("✅ Daten erfolgreich geladen")
        print(f"\n📈 Zeitreihen:")
        print(f"  Zeitschritte:  {len(ts):,}")
        print(f"  Variablen:     {ts.shape[1]}")
        print(f"  Zeitraum:      {ts.index.min()} bis {ts.index.max()}")
        
        print(f"\n🔑 Verfügbare Spalten ({len(ts.columns)}):")
        for i, col in enumerate(sorted(ts.columns), 1):
            print(f"  {i:2d}. {col}")
        
        data_loaded = True
        
    except Exception as e:
        print(f"❌ Fehler beim Laden: {e}")
        data_loaded = False
else:
    print("⚠️  Keine Ergebnisse zum Laden verfügbar")
    print("    Führe zuerst die Optimierung aus!")
    data_loaded = False

---
## 📈 Visualisierungen

In [ ]:
if data_loaded:
    # Plot 1: Elektrische Leistung
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # Strom
    ax = axes[0]
    ts[['P_buy', 'P_sell']].plot(ax=ax, linewidth=2, alpha=0.8)
    ax.set_title('⚡ Elektrische Leistung', fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('Leistung [MW]', fontsize=12)
    ax.set_xlabel('')
    ax.grid(alpha=0.3, linestyle='--')
    ax.legend(['Netzbezug', 'Einspeisung'], fontsize=11, loc='upper right')
    
    # Statistik einblenden
    stats_text = f"Peak: {ts['P_buy'].max():.1f} MW\nTotal: {ts['P_buy'].sum():.0f} MWh"
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Wärme
    ax = axes[1]
    heat_cols = [c for c in ts.columns if ('_Q' in c or '_Qth' in c) and c != 'Q_dump']
    if heat_cols:
        # Top 5 für Übersichtlichkeit
        top_heat = ts[heat_cols].sum().nlargest(5).index.tolist()
        ts[top_heat].plot(ax=ax, linewidth=2, alpha=0.8)
        ax.set_title('🔥 Thermische Leistung (Top 5 Erzeuger)', fontsize=16, fontweight='bold', pad=20)
    else:
        ax.text(0.5, 0.5, 'Keine thermischen Daten gefunden', 
                ha='center', va='center', transform=ax.transAxes, fontsize=14)
    
    ax.set_ylabel('Leistung [MW_th]', fontsize=12)
    ax.set_xlabel('Zeit', fontsize=12)
    ax.grid(alpha=0.3, linestyle='--')
    ax.legend(fontsize=10, loc='upper right')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("⏭️  Überspringe Plots (keine Daten geladen)")

In [ ]:
if data_loaded:
    # Plot 2: Speicher & Heat Pumps
    n_plots = 0
    if 'TES_SOC' in ts.columns:
        n_plots += 1
    hp_cols = [c for c in ts.columns if c.startswith('HP') and '_Q' in c]
    if hp_cols:
        n_plots += 1
    
    if n_plots > 0:
        fig, axes = plt.subplots(n_plots, 1, figsize=(16, 5*n_plots))
        if n_plots == 1:
            axes = [axes]
        
        plot_idx = 0
        
        # Speicher
        if 'TES_SOC' in ts.columns:
            ax = axes[plot_idx]
            ts['TES_SOC'].plot(ax=ax, linewidth=2.5, color='orange', alpha=0.8)
            ax.fill_between(ts.index, 0, ts['TES_SOC'], alpha=0.3, color='orange')
            ax.set_title('🔋 Thermischer Speicher - State of Charge', 
                        fontsize=16, fontweight='bold', pad=20)
            ax.set_ylabel('Energie [MWh]', fontsize=12)
            ax.grid(alpha=0.3, linestyle='--')
            
            # Lade-/Entladezyklen zählen
            if 'TES_Qc' in ts.columns and 'TES_Qd' in ts.columns:
                cycles = ((ts['TES_Qc'] > 0).astype(int).diff().fillna(0) > 0).sum()
                stats = f"Zyklen: {cycles}\nMax: {ts['TES_SOC'].max():.1f} MWh"
                ax.text(0.02, 0.98, stats, transform=ax.transAxes,
                       fontsize=10, verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
            plot_idx += 1
        
        # Heat Pumps
        if hp_cols:
            ax = axes[plot_idx]
            ts[hp_cols].plot(ax=ax, linewidth=2, alpha=0.8)
            ax.set_title('♨️  Wärmepumpen - Thermische Leistung', 
                        fontsize=16, fontweight='bold', pad=20)
            ax.set_ylabel('Leistung [MW_th]', fontsize=12)
            ax.set_xlabel('Zeit', fontsize=12)
            ax.grid(alpha=0.3, linestyle='--')
            ax.legend(fontsize=10, loc='upper right')
        
        plt.tight_layout()
        plt.show()
    else:
        print("ℹ️  Keine Speicher- oder HP-Daten zum Plotten")

---
## 📊 KPI-Analyse

In [ ]:
if data_loaded:
    print("\n" + "="*70)
    print("📊 KEY PERFORMANCE INDICATORS")
    print("="*70)
    
    # Kosten
    print(f"\n💰 Wirtschaftlichkeit:")
    print(f"  Gesamtkosten:        {res['costs']['OBJ_value_EUR']:>15,.0f} EUR")
    print(f"  Peak-Leistung:       {res['costs']['P_buy_peak_MW']:>15,.2f} MW")
    
    # Energie-Bilanz
    print(f"\n⚡ Elektrische Energie:")
    e_buy = ts['P_buy'].sum()
    e_sell = ts['P_sell'].sum()
    print(f"  Netzbezug:           {e_buy:>15,.0f} MWh")
    print(f"  Einspeisung:         {e_sell:>15,.0f} MWh")
    print(f"  Netto:               {(e_buy - e_sell):>15,.0f} MWh")
    
    # Wärme
    if 'Q_dump' in ts.columns:
        q_dump = ts['Q_dump'].sum()
        print(f"\n🔥 Thermische Energie:")
        print(f"  Wärme-Dump (Verlust): {q_dump:>14,.0f} MWh_th")
    
    # Komponenten-Auslastung
    print(f"\n🏭 Komponenten-Auslastung:")
    
    # Heat Pumps
    hp_cols = [c for c in ts.columns if c.startswith('HP') and c.endswith('_Q')]
    if hp_cols:
        print(f"\n  Wärmepumpen:")
        for col in sorted(hp_cols):
            hp_id = col.replace('_Q', '')
            avg = ts[col].mean()
            max_val = ts[col].max()
            hours_on = (ts[col] > 0.01).sum()
            print(f"    {hp_id:8s}: Ø {avg:6.2f} MW | Max {max_val:6.2f} MW | {hours_on:5d} h aktiv")
    
    # Generatoren
    gen_cols = [c for c in ts.columns if c.endswith('_Qth') and not c.startswith('HP')]
    if gen_cols:
        print(f"\n  Thermische Generatoren:")
        for col in sorted(gen_cols):
            gen_id = col.replace('_Qth', '')
            avg = ts[col].mean()
            max_val = ts[col].max()
            total = ts[col].sum()
            print(f"    {gen_id:8s}: Ø {avg:6.2f} MW | Max {max_val:6.2f} MW | {total:8,.0f} MWh_th")
    
    # Speicher-Performance
    if 'TES_SOC' in ts.columns:
        print(f"\n  Speicher:")
        print(f"    Max SOC:           {ts['TES_SOC'].max():>10.2f} MWh")
        print(f"    Ø SOC:             {ts['TES_SOC'].mean():>10.2f} MWh")
        if 'TES_Qc' in ts.columns:
            print(f"    Total geladen:     {ts['TES_Qc'].sum():>10,.0f} MWh")
        if 'TES_Qd' in ts.columns:
            print(f"    Total entladen:    {ts['TES_Qd'].sum():>10,.0f} MWh")
    
    print("\n" + "="*70)

---
## 🔍 Datenexploration

In [ ]:
if data_loaded:
    # DataFrame-Vorschau
    print("📋 Zeitreihen-Daten (erste 10 Zeilen):\n")
    display(ts.head(10))
    
    print("\n📊 Statistische Zusammenfassung:\n")
    display(ts.describe())

In [ ]:
if data_loaded:
    # Korrelationsmatrix (für Profis)
    numeric_cols = ts.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 1:
        fig, ax = plt.subplots(figsize=(12, 10))
        corr = ts[numeric_cols].corr()
        im = ax.imshow(corr, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
        
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=90, ha='right')
        ax.set_yticklabels(corr.columns)
        
        plt.colorbar(im, ax=ax)
        ax.set_title('🔗 Korrelationsmatrix', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()

---
## 💾 Export & Weiterverarbeitung

In [ ]:
if data_loaded:
    print("📁 Export-Informationen:\n")
    print(f"  Excel (komplett):  {res['scenario_xlsx']}")
    print(f"  Verzeichnis:       {res['outdir']}")
    print(f"\n📄 Enthaltene Sheets:")
    print(f"  - timeseries: Alle Zeitreihen-Variablen")
    print(f"  - costs:      Kosten-Breakdown")
    print(f"  - meta:       Config-Hash & Provenance")
    
    # Optional: CSV-Export
    csv_path = Path(res['outdir']) / 'timeseries.csv'
    ts.to_csv(csv_path)
    print(f"\n✅ CSV exportiert: {csv_path}")

---
## 🎯 Nächste Schritte

### Sensitivitätsanalysen:
- CO2-Preis variieren
- Komponenten aktivieren/deaktivieren  
- Kapazitäten anpassen

### Weitere Analysen:
- Jahresdauerlinie erstellen
- Monats-Aggregation
- COP-Entwicklung analysieren

### Reporting:
- Plots als PDF exportieren
- Automatische Berichte generieren
---